# Pipeline KYC — Extraction automatique d'information depuis les justificatifs d'identité

**Ce que fait ce notebook, étape par étape :**

1. Décompresse **sélectivement** `kyc_documents.zip` (un dossier par client) : seuls les 5 documents PDF utiles sont extraits, le reste de l'archive n'est pas touché.
2. Vérifie, pour chaque client, la présence de ces 5 documents et produit un **rapport d'existence** (CSV + Excel).
3. Identifie les **clients cibles** : ceux pour lesquels `JUSTIFICATIF IDENTITE.PDF` existe.
4. Pour ces clients, extrait automatiquement **toute l'information** contenue dans `JUSTIFICATIF IDENTITE.PDF` — document scanné, potentiellement multi-pages, multilingue (ou mélange de langues), parfois mal orienté ou de qualité médiocre — à l'aide du modèle multimodal **Qwen** déployé localement sur Domino.

**Hypothèses retenues (à valider de votre côté avant exécution en production)**

- Le modèle situé à `MODEL_PATH` est une variante **vision-langage** de Qwen (capable de lire des images), pas une variante texte seul. La section 7 vérifie automatiquement l'architecture déclarée dans `config.json` et vous alerte si ce n'est pas le cas.
- Le checkpoint étant quantifié en **FP8 fine-grained** (blocs 128×128), il est chargé via `transformers`, qui prend en charge nativement ce schéma (`FineGrainedFP8`). Le kernel de calcul associé, `kernels-community/finegrained-fp8`, est lu depuis la **copie locale disponible sur ModelHub** — aucun accès réseau sortant n'est nécessaire (section 7).
- Conformément à la demande, **seul `JUSTIFICATIF IDENTITE.PDF` est traité** dans cette première itération. La même mécanique (sections 8 à 10) est réutilisable telle quelle pour les 4 autres documents, en adaptant le prompt d'extraction.

**Rappels importants**

- 🔒 Ce pipeline manipule des données personnelles sensibles (KYC/LCB-FT). Vérifiez qu'il s'exécute dans un environnement conforme à la politique de gouvernance des données de votre établissement, et que les sorties (rapports, JSON) sont stockées à un emplacement dûment restreint. L'inférence se fait entièrement en local (aucun appel à une API externe).
- ⚠️ Un modèle de vision-langage peut se tromper ou halluciner, en particulier sur des scans de mauvaise qualité ou de l'écriture manuscrite. Les champs extraits automatiquement sont une **aide à la saisie à valider par un contrôle humain**, pas une source de vérité autonome pour une décision réglementaire.

## 1. Installation des librairies

Installées dans l'ordre suivant : traitement PDF/image → détection d'orientation → rapports/structuration → moteur d'inférence du modèle Qwen.

> ⚠️ **Dépendance système** : `pytesseract` s'appuie sur le binaire `tesseract-ocr` (utilisé ici uniquement pour la **détection d'orientation** des pages, pas pour l'extraction de texte). Sur une image Domino Debian/Ubuntu avec les droits suffisants :
> ```bash
> apt-get update && apt-get install -y tesseract-ocr
> ```
> Si vous n'avez pas les droits d'installation système, demandez à votre équipe plateforme d'ajouter ce paquet à l'environnement Domino, ou passez `USE_OSD_ROTATION = False` en section 8 (le reste du pipeline fonctionne à l'identique, seule la correction automatique de rotation est désactivée).

In [ ]:
# 1) Traitement PDF et image
%pip install --quiet PyMuPDF==1.26.1 pillow==11.0.0 opencv-python-headless==4.10.0.84 numpy==1.26.4

# 2) Détection d'orientation des scans (OSD) — nécessite aussi le binaire système tesseract-ocr (voir note ci-dessus)
%pip install --quiet pytesseract==0.3.13

# 3) Rapports et structuration des résultats
%pip install --quiet pandas==2.2.3 openpyxl==3.1.5 tqdm==4.66.5

# 4) Moteur d'inférence : transformers.
#    IMPORTANT : on ne fixe PAS `torch` ici. L'environnement Domino (DWS-GPU) fournit
#    déjà une build de PyTorch alignée sur son pilote/CUDA ; la réinstaller est le
#    meilleur moyen de casser l'environnement. On installe uniquement ce qui manque.
%pip install --quiet --upgrade "transformers>=4.57.0" "accelerate>=1.2.0"

# 5) Kernel FP8 fine-grained (quantification par blocs 128x128).
#    `transformers` charge ce kernel via la librairie `kernels`, dont il n'accepte
#    qu'une plage de versions précise (>=0.16.0, <0.17.0 — cf. KERNELS_MIN_VERSION /
#    KERNELS_MAX_VERSION dans transformers). Le kernel lui-même n'est PAS téléchargé :
#    il est lu depuis ModelHub via la variable LOCAL_KERNELS (voir section 3).
%pip install --quiet "kernels>=0.16.0,<0.17.0"

## 2. Imports et configuration du logging

In [ ]:
from __future__ import annotations

import io
import os
import re
import json
import zipfile
import logging
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pytesseract
import torch
import fitz  # PyMuPDF
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("kyc_pipeline")

## 3. Configuration

Tous les chemins et paramètres modifiables sont centralisés ici — adaptez-les à l'arborescence de votre projet Domino.

In [ ]:
# --------------------------------------------------------------------------
# Chemins
# --------------------------------------------------------------------------
ZIP_PATH = Path("./kyc_documents.zip")

WORKDIR = Path("./kyc_pipeline_output")
EXTRACT_DIR = WORKDIR / "documents_extraits"      # PDF cibles extraits, un sous-dossier par client
REPORTS_DIR = WORKDIR / "rapports"                # rapport d'existence, liste des clients cibles, synthèse
RESULTS_DIR = WORKDIR / "resultats_extraction"    # un JSON détaillé par client traité

for _d in (EXTRACT_DIR, REPORTS_DIR, RESULTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------------
# Modèle local (Domino ModelHub)
# --------------------------------------------------------------------------
MODEL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.8-27B-FP8/main"

# --------------------------------------------------------------------------
# Kernel FP8 fine-grained, disponible localement sur ModelHub.
#
# `transformers` charge ce kernel depuis le Hub sous le nom
# "kernels-community/finegrained-fp8" (version 4). La variable d'environnement
# LOCAL_KERNELS, lue par la librairie `kernels`, permet de rediriger ce nom vers
# une copie locale — indispensable ici, l'environnement n'ayant a priori pas
# d'accès sortant vers huggingface.co.
#
# Format attendu : "repo_id_1=chemin_1:repo_id_2=chemin_2"
# --------------------------------------------------------------------------
FP8_KERNEL_REPO_ID = "kernels-community/finegrained-fp8"
FP8_KERNEL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-kernels-community/finegrained-fp8/v4"

os.environ["LOCAL_KERNELS"] = f"{FP8_KERNEL_REPO_ID}={FP8_KERNEL_PATH}"

# Mode hors-ligne : empêche toute tentative d'appel sortant vers huggingface.co.
# Tout (modèle, processeur, kernel) est lu depuis le système de fichiers local.
# Passez à False si votre environnement dispose d'un accès réseau et que vous
# préférez laisser transformers compléter d'éventuels fichiers manquants.
OFFLINE_MODE = True
if OFFLINE_MODE:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"

# --------------------------------------------------------------------------
# Documents cibles à repérer/extraire dans le ZIP.
# Nom canonique -> motif de reconnaissance du nom de fichier (insensible à la
# casse, espaces multiples tolérés). "CARTON_SIGNATURE" tolère à la fois
# l'orthographe correcte (SIGNATURE) et la coquille du cahier des charges
# d'origine (SIGNATUTE), au cas où les deux variantes coexisteraient dans
# les dossiers réels.
# --------------------------------------------------------------------------
TARGET_FILES = {
    "JUSTIFICATIF_IDENTITE": r"JUSTIFICATIF\s+IDENTITE\.PDF",
    "JUSTIFICATIF_DOMICILE": r"JUSTIFICATIF\s+DOMICILE\.PDF",
    "CONVENTION_COMPTE":     r"CONVENTION\s+COMPTE\.PDF",
    "FATCA":                 r"FATCA\.PDF",
    "CARTON_SIGNATURE":      r"CARTON\s+SIGNAT(?:URE|UTE)\.PDF",
}
COMPILED_PATTERNS = {k: re.compile(v, re.IGNORECASE) for k, v in TARGET_FILES.items()}

# Document sur lequel porte cette première itération d'extraction.
# (Pour traiter un autre document dans une itération future, il suffit de
# changer cette valeur et d'adapter le prompt de la section 9.)
DOCUMENT_A_TRAITER = "JUSTIFICATIF_IDENTITE"

print("Configuration chargée :")
print(f"  ZIP source         : {ZIP_PATH.resolve()}")
print(f"  Dossier de travail : {WORKDIR.resolve()}")
print(f"  Modèle Qwen        : {MODEL_PATH}")

## 4. Étape 1 — Décompression sélective de l'archive

On parcourt `kyc_documents.zip` **sans tout décompresser** : seuls les fichiers dont le nom correspond à l'un des 5 documents cibles sont extraits, dans une arborescence propre `documents_extraits/<client_id>/<NOM_CANONIQUE>.pdf`.

L'identifiant client est déduit du **dossier parent immédiat** de chaque fichier dans l'archive — robuste à un éventuel dossier racine englobant (ex. `kyc_documents/<client_id>/...`).

In [ ]:
def match_canonical_name(filename: str) -> str | None:
    '''Retourne le nom canonique du document si `filename` correspond à
    l'un des documents cibles (regex, insensible à la casse), sinon None.'''
    filename = filename.strip()
    for canonical, pattern in COMPILED_PATTERNS.items():
        if pattern.fullmatch(filename):
            return canonical
    return None


def extract_target_files_from_zip(zip_path: Path, output_dir: Path):
    '''Extrait uniquement les documents cibles de l'archive ZIP, organisés
    par client.

    Retourne :
      - client_files : {client_id: {nom_canonique: chemin_extrait}}
      - unmatched_by_client : {client_id: [noms de fichiers non reconnus]}
      - n_entries : nombre total d'entrées fichier lues dans l'archive
    '''
    client_files: dict[str, dict[str, Path]] = defaultdict(dict)
    unmatched_by_client: dict[str, list[str]] = defaultdict(list)
    n_entries = 0

    if not zip_path.exists():
        raise FileNotFoundError(
            f"Archive introuvable : {zip_path.resolve()}. Vérifiez ZIP_PATH dans la configuration."
        )

    with zipfile.ZipFile(zip_path, "r") as zf:
        members = [m for m in zf.infolist() if not m.is_dir()]
        for info in tqdm(members, desc="Lecture de l'archive"):
            n_entries += 1
            parts = Path(info.filename).parts
            if len(parts) < 2:
                logger.warning(f"Fichier à la racine ignoré (pas de dossier client identifiable) : {info.filename}")
                continue

            client_id = parts[-2]
            filename = parts[-1]
            canonical = match_canonical_name(filename)

            if canonical is None:
                if filename:  # ignore les entrées de dossier vides
                    unmatched_by_client[client_id].append(filename)
                continue

            dest_path = output_dir / client_id / f"{canonical}.pdf"
            dest_path.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, open(dest_path, "wb") as dst:
                dst.write(src.read())
            client_files[client_id][canonical] = dest_path

    return client_files, unmatched_by_client, n_entries


client_files, unmatched_by_client, n_entries = extract_target_files_from_zip(ZIP_PATH, EXTRACT_DIR)

n_clients = len(set(client_files) | set(unmatched_by_client))
n_extracted = sum(len(v) for v in client_files.values())
logger.info(
    f"{n_entries} fichiers lus dans l'archive, {n_clients} dossiers clients détectés, "
    f"{n_extracted} documents cibles extraits vers {EXTRACT_DIR}"
)

## 5. Étape 2 — Rapport d'existence des documents par client

Pour chaque client détecté dans l'archive, on vérifie la présence de chacun des 5 documents cibles et on sauvegarde le résultat (CSV + Excel), exploitable directement par les équipes métier. Les éventuels fichiers présents dans un dossier mais non reconnus comme l'un des 5 documents cibles sont aussi listés, à titre de contrôle qualité (typo de nommage, document inattendu, etc.).

In [ ]:
def build_existence_report(client_files, unmatched_by_client) -> pd.DataFrame:
    all_clients = sorted(set(client_files.keys()) | set(unmatched_by_client.keys()))
    rows = []
    for client_id in all_clients:
        found = client_files.get(client_id, {})
        row = {"client_id": client_id}
        for canonical in TARGET_FILES:
            row[canonical] = canonical in found
        row["nb_documents_cibles_trouves"] = sum(row[c] for c in TARGET_FILES)
        row["nb_documents_cibles_attendus"] = len(TARGET_FILES)
        row["dossier_complet"] = row["nb_documents_cibles_trouves"] == len(TARGET_FILES)
        row["documents_manquants"] = ", ".join(c for c in TARGET_FILES if not row[c])
        row["fichiers_non_reconnus_dans_le_dossier"] = ", ".join(unmatched_by_client.get(client_id, []))
        rows.append(row)
    return pd.DataFrame(rows)


existence_report_df = build_existence_report(client_files, unmatched_by_client)

report_csv_path = REPORTS_DIR / "rapport_existence_documents.csv"
existence_report_df.to_csv(report_csv_path, index=False, encoding="utf-8-sig")

try:
    existence_report_df.to_excel(REPORTS_DIR / "rapport_existence_documents.xlsx", index=False)
except Exception as e:
    logger.warning(f"Export Excel ignoré ({e}) — le CSV reste disponible.")

logger.info(f"Rapport d'existence sauvegardé : {report_csv_path}")
print(
    f"\n{existence_report_df['dossier_complet'].sum()} / {len(existence_report_df)} "
    "dossiers complets (les 5 documents cibles présents)."
)
existence_report_df

## 6. Étape 3 — Identification des clients cibles

Un client est considéré **cible** pour cette itération dès lors que `JUSTIFICATIF IDENTITE.PDF` est présent dans son dossier — indépendamment de la présence des 4 autres documents (qui pourront être traités dans une itération ultérieure, cf. section 12).

In [ ]:
target_clients_df = existence_report_df.loc[
    existence_report_df["JUSTIFICATIF_IDENTITE"],
    ["client_id", "nb_documents_cibles_trouves", "dossier_complet"],
].reset_index(drop=True)

target_clients = target_clients_df["client_id"].tolist()

target_clients_csv_path = REPORTS_DIR / "clients_cibles.csv"
target_clients_df.to_csv(target_clients_csv_path, index=False, encoding="utf-8-sig")

logger.info(
    f"{len(target_clients)} client(s) cible(s) identifié(s) sur {len(existence_report_df)} "
    f"(JUSTIFICATIF IDENTITE présent). Liste sauvegardée : {target_clients_csv_path}"
)
target_clients_df

## 7. Étape 4 — Chargement du modèle Qwen (vision-langage) via `transformers`

**Pourquoi `transformers` et pas vLLM ?** `transformers` sait désormais charger nativement les checkpoints **FP8 fine-grained** (quantification par blocs 128×128, `quant_method: "fp8"` dans `config.json`) : c'est la quantification `FineGrainedFP8`. Pour cela il s'appuie sur un kernel externe nommé `kernels-community/finegrained-fp8`, normalement téléchargé depuis le Hub — mais dont **une copie locale est disponible sur ModelHub**. La variable `LOCAL_KERNELS` (définie en section 3) redirige le chargement vers cette copie, ce qui permet de fonctionner entièrement hors-ligne.

Cette voie évite le plantage rencontré au démarrage du moteur multiprocessus de vLLM (`TypeError: type 'array.array' is not subscriptable`), qui provient d'une syntaxe de typage réservée à Python ≥ 3.12 alors que l'environnement `DWS-GPU` tourne en Python 3.11.

**Contrepartie assumée :** `model.generate()` traite un document à la fois, là où vLLM aurait permis de batcher les requêtes. Pour un traitement séquentiel client par client, la différence de débit reste acceptable ; si le volume devient important, la piste vLLM (section 12) redevient pertinente.

Les deux cellules ci-dessous séparent volontairement **le diagnostic** (que voit-on réellement dans l'environnement ?) et **le chargement** (opération lourde, plusieurs minutes) — pour que le diagnostic reste consultable même si le chargement échoue.

In [ ]:
config_path = Path(MODEL_PATH) / "config.json"
if not config_path.exists():
    raise FileNotFoundError(f"config.json introuvable dans {MODEL_PATH} — vérifiez MODEL_PATH.")

with open(config_path, "r", encoding="utf-8") as f:
    model_config = json.load(f)

print("=== Informations sur le modèle local ===")
print(f"  architectures        : {model_config.get('architectures')}")
print(f"  model_type           : {model_config.get('model_type')}")
print(f"  quantization_config  : {model_config.get('quantization_config')}")

archs = [a.lower() for a in model_config.get("architectures", [])]
is_probably_vl = any(("vl" in a) or ("vision" in a) for a in archs) or "image-text-to-text" in json.dumps(model_config).lower()
if not is_probably_vl:
    logger.warning(
        "L'architecture déclarée ne semble PAS être une variante vision-langage de Qwen. "
        "Ce notebook suppose un modèle capable de lire des images (Qwen-VL / Qwen2-VL / "
        "Qwen2.5-VL / Qwen3-VL). Si ce n'est pas le cas, l'étape d'extraction (section 9) "
        "échouera ou donnera des résultats incohérents : vérifiez avec votre équipe "
        "plateforme quelle variante de Qwen a été déployée sur ModelHub."
    )

print("\n=== Ressources GPU visibles ===")
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if n_gpus:
    for i in range(n_gpus):
        name = torch.cuda.get_device_name(i)
        cap = torch.cuda.get_device_capability(i)
        mem_gb = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {name} | capability={cap} | mémoire={mem_gb:.1f} Go")
        if cap < (8, 9):
            logger.warning(
                f"  GPU {i} : capability {cap} < (8,9) — l'accélération matérielle FP8 native "
                "(Ada Lovelace/Hopper, ex. H100/L40S/L4) n'est probablement pas disponible sur ce "
                "GPU. Le chargement peut échouer ou être lent ; vérifiez avec votre équipe "
                "plateforme quel GPU est réellement alloué sur Domino."
            )
else:
    logger.warning(
        "Aucun GPU détecté par PyTorch — l'inférence sur un modèle de cette taille sur CPU "
        "seul serait extrêmement lente, voire impraticable. Vérifiez l'allocation de votre "
        "exécution Domino (hardware tier avec GPU)."
    )

# --- Diagnostics d'environnement -------------------------------------------
# Utile en particulier si la cellule suivante (chargement du modèle) échoue avec
# "Engine core initialization failed." : ce message générique ne montre PAS
# la vraie cause (elle est affichée juste au-dessus, dans la sortie de la
# cellule suivante). La cause la plus fréquente est un décalage de version
# entre les paquets installés par la cellule d'installation (section 1) et la pile
# CUDA/PyTorch déjà préconfigurée par votre environnement Domino — les
# informations ci-dessous permettent de repérer ce genre de décalage.
print("\n=== Kernel FP8 local (ModelHub) ===")
# `kernels` cherche un sous-dossier `build/<variante>` (ex. torch28-cxx11-cu126-x86_64-linux),
# où la variante doit correspondre à votre couple torch/CUDA. On liste ce qui est
# réellement présent : si aucune variante ne correspond, le chargement échouera avec
# une erreur peu explicite, autant le voir maintenant.
_kernel_path = Path(FP8_KERNEL_PATH)
if not _kernel_path.exists():
    logger.error(
        f"Chemin du kernel introuvable : {FP8_KERNEL_PATH}. Vérifiez FP8_KERNEL_PATH "
        "(section 3) auprès de votre équipe plateforme."
    )
else:
    print(f"  Chemin      : {_kernel_path}")
    _build_dir = _kernel_path / "build"
    if _build_dir.is_dir():
        _variants = sorted(p.name for p in _build_dir.iterdir() if p.is_dir())
        print(f"  Variantes   : {_variants}")
        if not _variants:
            logger.warning("  Le dossier 'build' existe mais ne contient aucune variante.")
    else:
        print(f"  Contenu     : {sorted(p.name for p in _kernel_path.iterdir())[:10]}")
        logger.warning(
            "  Pas de sous-dossier 'build/' : la structure attendue par `kernels` est "
            "<chemin>/build/<variante>. Le chargement tentera malgré tout d'importer "
            "le dossier directement (comportement de repli de get_local_kernel)."
        )
print(f"  LOCAL_KERNELS : {os.environ.get('LOCAL_KERNELS')}")

print("\n=== Versions installées ===")
import importlib
for _pkg in ("torch", "transformers", "kernels", "accelerate", "triton"):
    try:
        _mod = importlib.import_module(_pkg)
        print(f"  {_pkg:12s}: {getattr(_mod, '__version__', '?')}")
    except ImportError:
        print(f"  {_pkg:12s}: non installé")
print(f"  {'torch+cuda':12s}: {torch.version.cuda}  (version CUDA avec laquelle PyTorch a été compilé)")

print("\n=== nvidia-smi (driver, CUDA runtime, mémoire déjà utilisée) ===")
import subprocess
try:
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=20)
    print(smi.stdout if smi.returncode == 0 else f"nvidia-smi a échoué : {smi.stderr}")
except Exception as e:
    print(f"nvidia-smi indisponible : {e}")

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

MAX_IMAGES_PER_PROMPT = 8    # nombre max de pages-image envoyées en une seule requête
MAX_NEW_TOKENS = 3000        # longueur max de la réponse générée

# Vérification préalable : sans la librairie `kernels` dans la bonne plage de versions,
# transformers ne pourra pas charger le kernel FP8 et l'erreur remontée serait obscure.
try:
    from transformers.utils.import_utils import is_kernels_available
    if not is_kernels_available():
        logger.warning(
            "La librairie `kernels` est absente ou hors de la plage de versions acceptée "
            "par transformers (>=0.16.0, <0.17.0). Le chargement FP8 risque d'échouer : "
            "relancez la cellule d'installation (section 1) puis redémarrez le kernel Jupyter."
        )
except ImportError:
    pass  # fonction interne, absente de certaines versions : simple vérification de confort

logger.info(f"Chargement du processeur depuis {MODEL_PATH}...")
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)

logger.info(
    "Chargement du modèle (quantification FP8 détectée depuis config.json, kernel lu "
    "localement via LOCAL_KERNELS)... cela peut prendre plusieurs minutes."
)

try:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_PATH,
        dtype="auto",          # respecte le dtype déclaré dans config.json
        device_map="auto",     # répartit automatiquement sur les GPU disponibles (nécessite accelerate)
        trust_remote_code=True,
        # Pas de `quantization_config` à passer : le checkpoint est DÉJÀ quantifié,
        # transformers lit `quantization_config` depuis config.json et instancie
        # le quantifieur FineGrainedFP8 tout seul.
    )
except Exception:
    logger.error(
        "Échec du chargement du modèle. Pistes, dans l'ordre de probabilité :\n"
        "  1. Kernel FP8 introuvable ou variante incompatible → voir le diagnostic "
        "'Kernel FP8 local' de la cellule précédente (la variante doit correspondre "
        "à votre couple torch/CUDA).\n"
        "  2. Mémoire GPU insuffisante → vérifiez la sortie nvidia-smi ci-dessus.\n"
        "  3. Architecture non reconnue → vérifiez que votre version de transformers "
        "est assez récente pour l'architecture affichée dans le diagnostic."
    )
    raise

model.eval()
logger.info(f"Modèle chargé. Répartition mémoire : {getattr(model, 'hf_device_map', 'device unique')}")

## 8. Étape 5 — Prétraitement des scans (conversion, orientation, qualité)

Trois fonctions, volontairement "légères" (CPU, rapides) — c'est le modèle Qwen, plus robuste et multilingue, qui porte l'essentiel de la charge de compréhension du document, pas un moteur OCR classique :

1. **`pdf_to_images`** — convertit chaque page du PDF en image haute résolution (300 DPI par défaut, suffisant pour la petite écriture d'une pièce d'identité).
2. **`detect_and_fix_rotation`** — utilise Tesseract en mode **OSD** (Orientation and Script Detection — *pas* une extraction de texte complète) pour détecter si une page est tournée de 90°/180°/270° et la remettre à l'endroit. Fonctionne indépendamment de la langue du document. La confiance de détection (`confiance_osd`) est conservée : sur des pages peu chargées en texte, cette confiance peut être faible — en-dessous du seuil `MIN_OSD_CONFIDENCE`, la rotation n'est **pas** appliquée automatiquement, pour éviter de dégrader une page déjà correcte, et le cas est signalé pour revue.
3. **`enhance_scan`** — égalisation adaptative du contraste (CLAHE), volontairement douce pour ne pas dégrader l'écriture manuscrite (pas de débruitage agressif ni de binarisation destructive).

> **Limite connue** : un retournement *miroir* (image inversée gauche-droite, différent d'une simple rotation) n'est pas corrigé automatiquement ici. Si ce cas se présente dans vos scans, une étape dédiée serait nécessaire.

In [ ]:
USE_OSD_ROTATION = True     # passez à False si tesseract-ocr n'est pas installé sur votre environnement
MIN_OSD_CONFIDENCE = 1.0    # en-dessous de ce seuil, la rotation détectée est jugée peu fiable : elle n'est pas appliquée mais reste tracée
SCAN_DPI = 300


def pdf_to_images(pdf_path: Path, dpi: int = SCAN_DPI) -> list[Image.Image]:
    '''Convertit chaque page d'un PDF en image PIL RGB haute résolution.'''
    images = []
    with fitz.open(pdf_path) as doc:
        zoom = dpi / 72  # 72 dpi = résolution native d'une page PDF
        matrix = fitz.Matrix(zoom, zoom)
        for page in doc:
            pix = page.get_pixmap(matrix=matrix)
            img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
            images.append(img)
    return images


def detect_and_fix_rotation(pil_image: Image.Image) -> tuple[Image.Image, dict]:
    '''Détecte et corrige une rotation de 90/180/270° via Tesseract OSD.
    Retourne l'image (corrigée ou non) et des métadonnées de diagnostic.'''
    meta = {"angle_detecte": 0, "confiance_osd": None, "rotation_appliquee": False, "osd_erreur": None}
    if not USE_OSD_ROTATION:
        return pil_image, meta

    try:
        osd = pytesseract.image_to_osd(pil_image, output_type=pytesseract.Output.DICT)
        angle = int(osd.get("rotate", 0))
        conf = float(osd.get("orientation_conf", 0.0))
        meta["angle_detecte"] = angle
        meta["confiance_osd"] = conf
    except pytesseract.TesseractError as e:
        meta["osd_erreur"] = str(e)
        logger.debug(f"OSD Tesseract a échoué : {e}")
        return pil_image, meta

    if angle != 0 and conf >= MIN_OSD_CONFIDENCE:
        # PIL rotate() tourne dans le sens anti-horaire ; l'angle Tesseract indique
        # la rotation horaire nécessaire pour remettre le texte droit -> on inverse le signe.
        pil_image = pil_image.rotate(-angle, expand=True)
        meta["rotation_appliquee"] = True
    elif angle != 0:
        logger.debug(f"Rotation de {angle}° détectée mais confiance faible ({conf}) -> non appliquée")

    return pil_image, meta


def enhance_scan(pil_image: Image.Image) -> Image.Image:
    '''Égalisation adaptative légère du contraste (CLAHE sur le canal de
    luminance). Volontairement douce pour préserver l'écriture manuscrite.'''
    img = np.array(pil_image.convert("RGB"))
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge((l, a, b))
    return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))


def preprocess_scan(pil_image: Image.Image) -> tuple[Image.Image, dict]:
    '''Pipeline complet de prétraitement d'une page scannée.'''
    image, rotation_meta = detect_and_fix_rotation(pil_image)
    image = enhance_scan(image)
    return image, rotation_meta

## 9. Étape 6 — Prompt d'extraction et appel au modèle

Le prompt demande explicitement au modèle de :
- traiter toutes les pages fournies comme un **seul et même document** (recto/verso, ou plusieurs pages d'un même justificatif) ;
- gérer n'importe quelle langue ou mélange de langues **sans traduire** les valeurs (noms, numéros, dates transcrits tels qu'écrits) ;
- ne **jamais inventer** une information non visible (`null` plutôt que de deviner) ;
- signaler les zones illisibles plutôt que de les deviner ;
- répondre en JSON strict, avec un champ `texte_brut` de repli contenant toute la transcription (utile pour un contrôle humain même si le JSON structuré est incomplet).

`extract_identity_info_chunked` découpe automatiquement en plusieurs appels les documents dépassant `MAX_IMAGES_PER_PROMPT` pages, puis fusionne les résultats.

In [ ]:
EXTRACTION_PROMPT = '''Tu es un assistant spécialisé dans la lecture de pièces justificatives d'identité pour un dossier KYC bancaire.

Les images fournies sont les pages successives d'UN SEUL ET MÊME document d'identité (par exemple recto et verso d'une carte, ou plusieurs pages d'un passeport/permis). Le document peut être rédigé en français, en arabe, en anglais, ou dans un mélange de plusieurs langues — traite toutes les langues sans les traduire.

Consignes strictes :
1. Transcris les valeurs (noms, prénoms, numéros, dates, adresses, lieux) exactement telles qu'elles apparaissent, sans traduire ni reformater. Conserve la graphie d'origine, y compris les caractères non latins.
2. Ne devine ni n'invente JAMAIS une information qui n'est pas clairement visible. Si un champ est absent, illisible ou non applicable, indique `null`.
3. Si une portion de texte est partiellement illisible (mauvaise qualité de scan, écriture manuscrite ambiguë), transcris ce que tu peux lire et signale-le dans "commentaires_qualite".
4. Réponds UNIQUEMENT avec un objet JSON valide, sans texte avant ou après, sans balises de code, selon EXACTEMENT le schéma suivant :

{
  "type_document": "carte d'identité nationale / passeport / permis de conduire / autre (préciser) / null",
  "nom": "string ou null",
  "prenom": "string ou null",
  "date_naissance": "string (telle qu'écrite sur le document) ou null",
  "lieu_naissance": "string ou null",
  "sexe": "string ou null",
  "nationalite": "string ou null",
  "numero_document": "string ou null",
  "date_delivrance": "string ou null",
  "date_expiration": "string ou null",
  "autorite_delivrance": "string ou null",
  "adresse_si_presente": "string ou null",
  "langues_detectees": ["liste des langues identifiées sur le document"],
  "nombre_de_pages_analysees": <entier>,
  "commentaires_qualite": "remarques sur la lisibilité, rotation, pages illisibles, etc. ou null",
  "texte_brut": "transcription complète et brute de tout le texte visible sur toutes les pages, dans l'ordre, séparée par ' | ' entre pages"
}
'''


def build_extraction_messages(images: list[Image.Image], prompt_text: str) -> list[dict]:
    '''Construit la conversation multimodale au format attendu par le chat template.
    Une entrée {"type": "image"} par page : le template insère un marqueur d'image
    à chaque position, et ces marqueurs doivent correspondre exactement, en nombre
    et en ordre, aux images passées ensuite au processeur.'''
    content = [{"type": "image"} for _ in images]
    content.append({"type": "text", "text": prompt_text})
    return [{"role": "user", "content": content}]


def safe_json_parse(text: str) -> tuple[dict, bool]:
    '''Extrait et parse un objet JSON depuis la réponse du modèle, même si
    celle-ci est entourée de texte ou de balises de code. Retourne (dict, ok).'''
    text = text.strip()
    try:
        return json.loads(text), True
    except json.JSONDecodeError:
        pass

    fence_match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence_match:
        try:
            return json.loads(fence_match.group(1)), True
        except json.JSONDecodeError:
            pass

    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start:end + 1]), True
        except json.JSONDecodeError:
            pass

    return {"_raw_text": text}, False


@torch.inference_mode()
def extract_identity_info(images: list[Image.Image]) -> dict:
    '''Envoie les pages d'un justificatif d'identité au modèle et retourne
    le résultat structuré. En cas de réponse non-JSON, le texte brut est
    conservé dans '_raw_text' et '_parsing_ok' vaut False.'''
    messages = build_extraction_messages(images, EXTRACTION_PROMPT)

    # 1) Le chat template produit le prompt textuel avec les marqueurs d'image.
    prompt_text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # 2) Le processeur encode texte + images ensemble (redimensionnement et
    #    normalisation des images gérés en interne par le processeur du modèle).
    inputs = processor(text=[prompt_text], images=images, return_tensors="pt")
    inputs = inputs.to(model.device)

    # 3) Génération déterministe (do_sample=False) : on veut une extraction
    #    reproductible d'un document à l'autre, pas de la créativité.
    generated = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
    )

    # 4) On ne décode que les tokens NOUVEAUX : `generated` contient le prompt
    #    complet en préfixe, qu'il faut retirer avant de parser la réponse.
    trimmed = generated[:, inputs["input_ids"].shape[1]:]
    raw_text = processor.batch_decode(trimmed, skip_special_tokens=True)[0]

    parsed, parsing_ok = safe_json_parse(raw_text)
    parsed["_parsing_ok"] = parsing_ok
    if not parsing_ok:
        logger.warning("Réponse du modèle non parsable en JSON strict — texte brut conservé pour revue manuelle.")
    return parsed


def extract_identity_info_chunked(images: list[Image.Image]) -> dict:
    '''Comme extract_identity_info, mais découpe automatiquement en plusieurs
    appels si le document compte plus de MAX_IMAGES_PER_PROMPT pages, puis
    fusionne les résultats (les champs structurés du premier bloc non-vide
    sont conservés, le texte_brut de tous les blocs est concaténé).'''
    if len(images) <= MAX_IMAGES_PER_PROMPT:
        return extract_identity_info(images)

    logger.info(f"Document de {len(images)} pages > {MAX_IMAGES_PER_PROMPT} : traitement par lots.")
    chunks = [images[i:i + MAX_IMAGES_PER_PROMPT] for i in range(0, len(images), MAX_IMAGES_PER_PROMPT)]
    chunk_results = [extract_identity_info(chunk) for chunk in chunks]

    merged: dict = {}
    for res in chunk_results:
        for key, value in res.items():
            if key == "texte_brut":
                continue
            if merged.get(key) in (None, "", []) and value not in (None, "", []):
                merged[key] = value
    merged["texte_brut"] = " | ".join(r.get("texte_brut", "") or "" for r in chunk_results)
    merged["_parsing_ok"] = all(r.get("_parsing_ok", False) for r in chunk_results)
    merged["_traite_par_lots"] = True
    return merged

## 10. Étape 7 — Exécution du pipeline sur les clients cibles

Pour chaque client cible : conversion du PDF en images → prétraitement (orientation + contraste) → extraction par le modèle → sauvegarde d'un JSON individuel.

Le traitement est **résilient** : une erreur sur un client (PDF corrompu, page illisible, etc.) est consignée et n'interrompt pas le traitement des autres clients. Il est également **reprenable** : si le notebook est relancé, les clients déjà traités avec succès (JSON déjà présent dans `resultats_extraction/`) sont réutilisés sans repasser par le modèle.

In [ ]:
SKIP_ALREADY_PROCESSED = True  # reprise : ignore les clients déjà traités avec succès lors d'une exécution précédente


def process_client_identity_document(client_id: str, pdf_path: Path) -> dict:
    '''Traite le justificatif d'identité d'un client et retourne un
    dictionnaire de résultat (incluant le statut du traitement).'''
    result = {
        "client_id": client_id,
        "fichier_source": str(pdf_path),
        "date_traitement": datetime.now().isoformat(timespec="seconds"),
        "statut": "succes",
        "erreur": None,
    }
    try:
        pages = pdf_to_images(pdf_path)
        if not pages:
            raise ValueError("Le PDF ne contient aucune page exploitable.")

        processed_pages, rotation_metas = [], []
        for page in pages:
            processed_page, rotation_meta = preprocess_scan(page)
            processed_pages.append(processed_page)
            rotation_metas.append(rotation_meta)

        extraction = extract_identity_info_chunked(processed_pages)

        confidences = [m["confiance_osd"] for m in rotation_metas if m["confiance_osd"] is not None]
        result["nombre_pages"] = len(pages)
        result["rotation_appliquee_sur"] = sum(m["rotation_appliquee"] for m in rotation_metas)
        result["confiance_orientation_min"] = min(confidences) if confidences else None
        result["extraction"] = extraction

    except Exception as e:
        result["statut"] = "erreur"
        result["erreur"] = f"{type(e).__name__}: {e}"
        result["extraction"] = None
        logger.error(f"[{client_id}] Échec du traitement : {result['erreur']}")

    return result


all_results = []
for client_id in tqdm(target_clients, desc="Extraction JUSTIFICATIF IDENTITE"):
    result_path = RESULTS_DIR / f"{client_id}.json"

    if SKIP_ALREADY_PROCESSED and result_path.exists():
        with open(result_path, "r", encoding="utf-8") as f:
            existing_result = json.load(f)
        if existing_result.get("statut") == "succes":
            all_results.append(existing_result)
            continue

    pdf_path = client_files.get(client_id, {}).get(DOCUMENT_A_TRAITER)
    if pdf_path is None:
        logger.error(f"[{client_id}] Incohérence inattendue : document cible introuvable, client ignoré.")
        continue

    result = process_client_identity_document(client_id, pdf_path)
    all_results.append(result)

    with open(result_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

n_success = sum(r["statut"] == "succes" for r in all_results)
logger.info(f"Traitement terminé : {n_success}/{len(all_results)} client(s) traité(s) avec succès.")

## 11. Étape 8 — Synthèse consolidée et contrôle qualité

On aplatit les résultats structurés en un tableau unique (un client par ligne) pour revue rapide par les équipes métier, puis on affiche un échantillon (image prétraitée + JSON extrait) pour un contrôle visuel rapide.

In [ ]:
def flatten_result(result: dict) -> dict:
    flat = {
        "client_id": result["client_id"],
        "statut": result["statut"],
        "erreur": result["erreur"],
        "nombre_pages": result.get("nombre_pages"),
        "rotation_appliquee_sur": result.get("rotation_appliquee_sur"),
        "confiance_orientation_min": result.get("confiance_orientation_min"),
    }
    extraction = result.get("extraction") or {}
    champs = [
        "type_document", "nom", "prenom", "date_naissance", "lieu_naissance",
        "sexe", "nationalite", "numero_document", "date_delivrance",
        "date_expiration", "autorite_delivrance", "adresse_si_presente",
        "langues_detectees", "commentaires_qualite",
    ]
    for champ in champs:
        val = extraction.get(champ)
        flat[champ] = ", ".join(val) if isinstance(val, list) else val
    flat["extraction_json_valide"] = extraction.get("_parsing_ok")
    return flat


synthese_df = pd.DataFrame([flatten_result(r) for r in all_results])

# --- Format 1 : JSON --------------------------------------------------------
# Source de vérité, sans perte : conserve la structure imbriquée, les listes
# (langues détectées), les valeurs nulles et le texte brut intégral.
#   - un fichier par client dans RESULTS_DIR (déjà écrit à l'étape 7)
#   - un fichier consolidé regroupant tous les clients, pratique pour une
#     reprise programmatique en aval (alimentation d'un référentiel, etc.)
json_consolide_path = REPORTS_DIR / "extraction_identite_complet.json"
with open(json_consolide_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "document_traite": DOCUMENT_A_TRAITER,
            "date_execution": datetime.now().isoformat(timespec="seconds"),
            "nombre_clients": len(all_results),
            "resultats": all_results,
        },
        f,
        ensure_ascii=False,   # conserve les accents et les caractères non latins tels quels
        indent=2,
    )

# --- Format 2 : CSV / Excel -------------------------------------------------
# Vue aplatie (une ligne par client, une colonne par champ) destinée à la revue
# humaine et au contrôle qualité. Aplatie = les structures imbriquées sont
# converties en texte : c'est le JSON ci-dessus qui fait foi en cas de doute.
synthese_csv_path = REPORTS_DIR / "synthese_extraction_identite.csv"
# utf-8-sig : Excel (Windows) affiche correctement les accents grâce au BOM.
synthese_df.to_csv(synthese_csv_path, index=False, encoding="utf-8-sig")
try:
    synthese_df.to_excel(REPORTS_DIR / "synthese_extraction_identite.xlsx", index=False)
except Exception as e:
    logger.warning(f"Export Excel ignoré ({e}).")

logger.info(
    f"Résultats sauvegardés :\n"
    f"  - JSON par client   : {RESULTS_DIR}/\n"
    f"  - JSON consolidé    : {json_consolide_path}\n"
    f"  - CSV               : {synthese_csv_path}\n"
    f"  - Excel             : {REPORTS_DIR / 'synthese_extraction_identite.xlsx'}"
)
synthese_df

In [ ]:
# Aperçu qualité : afficher la première page (prétraitée) et le JSON extrait
# pour le premier client traité avec succès, à des fins de contrôle visuel rapide.
premier_succes = next((r for r in all_results if r["statut"] == "succes"), None)

if premier_succes:
    apercu_pages = pdf_to_images(Path(premier_succes["fichier_source"]))
    apercu_page, _ = preprocess_scan(apercu_pages[0])
    display(apercu_page)
    print(json.dumps(premier_succes["extraction"], ensure_ascii=False, indent=2))
else:
    print("Aucun traitement réussi à afficher.")

## 12. Notes finales, limites et prochaines étapes

**Limites connues**

- La détection d'orientation (Tesseract OSD) est plus fiable sur des pages contenant un volume de texte substantiel ; sur des documents très épurés, la confiance peut être plus faible — c'est pourquoi `confiance_orientation_min` est reporté dans la synthèse, pour trier/prioriser les dossiers à vérifier manuellement.
- Un retournement miroir (image inversée gauche-droite, distinct d'une simple rotation) n'est pas corrigé automatiquement par ce notebook.
- Les résultats extraits par le modèle doivent être **validés par un contrôle humain** avant toute utilisation réglementaire — voir l'avertissement en introduction.

**Prochaines étapes suggérées**

- Répliquer les sections 8 à 11 pour les 4 autres documents (`JUSTIFICATIF_DOMICILE`, `CONVENTION_COMPTE`, `FATCA`, `CARTON_SIGNATURE`), avec un prompt adapté à chacun.
- Pour un volume important de dossiers, envisager le traitement par lots (voir la piste vLLM en fin de section) plutôt qu'un client à la fois, afin de maximiser le débit.
- Mettre en place un tableau de bord de suivi (taux de succès, taux de confiance d'orientation faible, dossiers à revoir manuellement).

**Formats de sortie produits**

| Fichier | Format | Usage |
|---|---|---|
| `resultats_extraction/<client_id>.json` | JSON | Résultat brut par client, sans perte (structure imbriquée, texte brut intégral) |
| `rapports/extraction_identite_complet.json` | JSON | Tous les clients en un seul fichier, pour reprise programmatique en aval |
| `rapports/synthese_extraction_identite.csv` | CSV | Vue aplatie, une ligne par client — revue humaine |
| `rapports/synthese_extraction_identite.xlsx` | Excel | Idem, pour diffusion aux équipes métier |

Le JSON fait foi : le CSV/Excel est une vue aplatie (listes converties en texte) destinée à la lecture.

**Piste alternative : vLLM (pour un volume important)**

`transformers` traite un document à la fois. Si le volume de dossiers devient important, vLLM permet d'envoyer plusieurs conversations en un seul appel (`llm.chat` accepte une liste), avec un gain de débit substantiel. Attention toutefois : c'est cette voie qui a échoué dans l'environnement `DWS-GPU` (Python 3.11) avec `TypeError: type 'array.array' is not subscriptable`, une syntaxe de typage réservée à Python ≥ 3.12. Si vous souhaitez y revenir, la piste la plus prometteuse est de faire préparer par votre équipe plateforme un environnement Domino en **Python 3.12**, où ce problème ne se pose plus.